# Notebook 04: Fine-tuning Protein Language Models (ESM-2)

**Lecture 10: Neural Networks and Deep Learning**  
**Center for Neuroscience and Cell Biology, University of Coimbra**

---

## Introduction

In this notebook, we'll learn how to fine-tune a pre-trained protein language model (ESM-2) for a downstream task. ESM-2 (Evolutionary Scale Modeling 2) is a transformer-based model trained on millions of protein sequences that has learned to represent protein structure and function.

**Learning Objectives:**
- Understand transfer learning in the context of protein language models
- Learn to load and use pre-trained ESM-2 models
- Fine-tune ESM-2 for protein function prediction
- Interpret model predictions and embeddings

**Applications:**
- Protein function prediction
- Variant effect prediction
- Protein engineering
- Drug target identification

## 1. Setup and Installation

First, let's install the required packages. ESM-2 is available through the `fair-esm` package from Meta AI.

In [ ]:
# Install required packages (uncomment to run)
# !pip install fair-esm torch biopython scikit-learn pandas matplotlib seaborn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import esm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Understanding ESM-2

ESM-2 is a transformer-based protein language model trained on evolutionary data:

**Key Features:**
- Pre-trained on 250M+ protein sequences from UniRef
- Available in multiple sizes: 8M, 35M, 150M, 650M, 3B, 15B parameters
- Captures evolutionary patterns and structural information
- State-of-the-art performance on protein prediction tasks

**Transfer Learning Strategy:**
1. **Feature Extraction:** Use ESM-2 embeddings as input to a classifier
2. **Fine-tuning:** Update ESM-2 weights for specific task
3. **Hybrid:** Freeze early layers, fine-tune later layers

## 3. Load Pre-trained ESM-2 Model

Let's load a pre-trained ESM-2 model. We'll use the `esm2_t12_35M_UR50D` model (35M parameters, 12 layers) which balances performance and computational requirements.

In [ ]:
# Load pre-trained ESM-2 model and alphabet
print("Loading ESM-2 model...")
model_name = "esm2_t12_35M_UR50D"
esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
esm_model = esm_model.to(device)
esm_model.eval()  # Set to evaluation mode initially

batch_converter = alphabet.get_batch_converter()

print(f"Model: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in esm_model.parameters()):,}")
print(f"Number of layers: {esm_model.num_layers}")
print(f"Embedding dimension: {esm_model.embed_dim}")

## 4. Create Synthetic Protein Dataset

For demonstration, we'll create a synthetic dataset of protein sequences with binary labels (e.g., membrane vs. soluble proteins).

In [ ]:
# Create synthetic protein sequences
# In practice, you would load real protein sequences with functional annotations

def generate_protein_sequence(length: int, gc_content: float = 0.5) -> str:
    """Generate a random protein sequence."""
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    return ''.join(np.random.choice(list(amino_acids), size=length))

# Generate synthetic dataset
np.random.seed(42)
n_samples = 200

# Class 0: Short, hydrophilic-rich sequences (soluble proteins)
class_0_sequences = [
    generate_protein_sequence(np.random.randint(50, 150))
    for _ in range(n_samples // 2)
]

# Class 1: Longer, hydrophobic-rich sequences (membrane proteins)
class_1_sequences = [
    generate_protein_sequence(np.random.randint(150, 300))
    for _ in range(n_samples // 2)
]

# Create DataFrame
df = pd.DataFrame({
    'sequence': class_0_sequences + class_1_sequences,
    'label': [0] * len(class_0_sequences) + [1] * len(class_1_sequences),
    'protein_id': [f'protein_{i:03d}' for i in range(n_samples)]
})

# Add sequence length feature
df['seq_length'] = df['sequence'].apply(len)

print(f"Dataset created: {len(df)} sequences")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nSequence length statistics:")
print(df.groupby('label')['seq_length'].describe())

# Display sample sequences
print("\nSample sequences:")
print(df.head())

In [ ]:
# Visualize dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Soluble (0)', 'Membrane (1)'], rotation=0)

# Sequence length distribution
for label in [0, 1]:
    subset = df[df['label'] == label]['seq_length']
    axes[1].hist(subset, bins=20, alpha=0.6, label=f'Class {label}')
axes[1].set_title('Sequence Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sequence Length')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Create PyTorch Dataset and DataLoader

In [ ]:
class ProteinDataset(Dataset):
    """Custom Dataset for protein sequences."""
    
    def __init__(self, sequences: List[str], labels: List[int], protein_ids: List[str]):
        self.sequences = sequences
        self.labels = labels
        self.protein_ids = protein_ids
    
    def __len__(self) -> int:
        return len(self.sequences)
    
    def __getitem__(self, idx: int) -> Tuple[str, str, int]:
        return self.protein_ids[idx], self.sequences[idx], self.labels[idx]

# Split data
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

# Create datasets
train_dataset = ProteinDataset(
    train_df['sequence'].tolist(),
    train_df['label'].tolist(),
    train_df['protein_id'].tolist()
)

val_dataset = ProteinDataset(
    val_df['sequence'].tolist(),
    val_df['label'].tolist(),
    val_df['protein_id'].tolist()
)

test_dataset = ProteinDataset(
    test_df['sequence'].tolist(),
    test_df['label'].tolist(),
    test_df['protein_id'].tolist()
)

# Create DataLoaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 6. Define Classification Model

We'll create a classifier that uses ESM-2 embeddings. We have two options:
1. **Feature Extraction:** Freeze ESM-2 weights, only train classifier
2. **Fine-tuning:** Update ESM-2 weights along with classifier

In [ ]:
class ESMProteinClassifier(nn.Module):
    """Protein classifier using ESM-2 embeddings."""
    
    def __init__(self, esm_model, n_classes: int = 2, freeze_esm: bool = False):
        super().__init__()
        self.esm_model = esm_model
        self.freeze_esm = freeze_esm
        
        # Freeze ESM-2 if using feature extraction
        if freeze_esm:
            for param in self.esm_model.parameters():
                param.requires_grad = False
        
        # Classification head
        hidden_dim = esm_model.embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, tokens):
        # Get ESM-2 embeddings
        with torch.set_grad_enabled(not self.freeze_esm):
            results = self.esm_model(tokens, repr_layers=[self.esm_model.num_layers])
            embeddings = results["representations"][self.esm_model.num_layers]
        
        # Use mean pooling over sequence length (excluding special tokens)
        # tokens: [batch_size, seq_len], embeddings: [batch_size, seq_len, embed_dim]
        mask = (tokens != alphabet.padding_idx).float().unsqueeze(-1)  # [batch, seq_len, 1]
        embeddings = embeddings * mask  # Zero out padding
        pooled = embeddings.sum(dim=1) / mask.sum(dim=1)  # Mean pooling
        
        # Classification
        logits = self.classifier(pooled)
        return logits

# Create model with frozen ESM-2 (feature extraction)
model = ESMProteinClassifier(esm_model, n_classes=2, freeze_esm=True)
model = model.to(device)

print(f"Model created with freeze_esm=True (feature extraction mode)")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Training Setup

In [ ]:
# Training hyperparameters
n_epochs = 10
learning_rate = 1e-3

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

## 8. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (protein_ids, sequences, labels) in enumerate(dataloader):
        # Prepare batch for ESM-2
        batch_data = [(pid, seq) for pid, seq in zip(protein_ids, sequences)]
        _, _, tokens = batch_converter(batch_data)
        tokens = tokens.to(device)
        labels = torch.tensor(labels).to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(tokens)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 5 == 0:
            print(f"  Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for protein_ids, sequences, labels in dataloader:
            # Prepare batch
            batch_data = [(pid, seq) for pid, seq in zip(protein_ids, sequences)]
            _, _, tokens = batch_converter(batch_data)
            tokens = tokens.to(device)
            labels = torch.tensor(labels).to(device)
            
            # Forward pass
            outputs = model(tokens)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Training loop
print("Starting training...\n")
best_val_loss = float('inf')

for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}/{n_epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/tmp/best_esm_classifier.pt')
    
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}\n")

print("Training complete!")

## 9. Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('/tmp/best_esm_classifier.pt'))
model.eval()

# Predict on test set
all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for protein_ids, sequences, labels in test_loader:
        # Prepare batch
        batch_data = [(pid, seq) for pid, seq in zip(protein_ids, sequences)]
        _, _, tokens = batch_converter(batch_data)
        tokens = tokens.to(device)
        
        # Predict
        outputs = model(tokens)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels)
        all_probabilities.extend(probabilities.cpu().numpy())

# Calculate metrics
test_acc = accuracy_score(all_labels, all_predictions)
print(f"Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, target_names=['Soluble', 'Membrane']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Soluble', 'Membrane'],
            yticklabels=['Soluble', 'Membrane'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 11. Extract and Visualize Embeddings

Let's extract ESM-2 embeddings and visualize them using t-SNE.

In [ ]:
from sklearn.manifold import TSNE

# Extract embeddings for test set
embeddings_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for protein_ids, sequences, labels in test_loader:
        # Prepare batch
        batch_data = [(pid, seq) for pid, seq in zip(protein_ids, sequences)]
        _, _, tokens = batch_converter(batch_data)
        tokens = tokens.to(device)
        
        # Get embeddings
        results = esm_model(tokens, repr_layers=[esm_model.num_layers])
        embeddings = results["representations"][esm_model.num_layers]
        
        # Mean pooling
        mask = (tokens != alphabet.padding_idx).float().unsqueeze(-1)
        embeddings = embeddings * mask
        pooled = embeddings.sum(dim=1) / mask.sum(dim=1)
        
        embeddings_list.append(pooled.cpu().numpy())
        labels_list.extend(labels)

# Concatenate all embeddings
embeddings_array = np.vstack(embeddings_list)
labels_array = np.array(labels_list)

print(f"Embeddings shape: {embeddings_array.shape}")

# t-SNE visualization
print("Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings_array) - 1))
embeddings_2d = tsne.fit_transform(embeddings_array)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=labels_array, cmap='viridis', 
                     alpha=0.6, s=100, edgecolors='k')
plt.colorbar(scatter, label='Class', ticks=[0, 1])
plt.title('t-SNE Visualization of ESM-2 Protein Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=l)
                   for c, l in zip(['purple', 'yellow'], ['Soluble', 'Membrane'])])
plt.grid(True, alpha=0.3)
plt.show()

## 12. Predict on New Sequences

Let's demonstrate how to use the trained model to predict on new protein sequences.

In [ ]:
def predict_protein(sequence: str, model, batch_converter, device) -> Dict:
    """Predict function for a single protein sequence."""
    model.eval()
    
    # Prepare input
    batch_data = [('protein', sequence)]
    _, _, tokens = batch_converter(batch_data)
    tokens = tokens.to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(tokens)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_class = torch.argmax(probabilities).item()
    
    return {
        'predicted_class': predicted_class,
        'predicted_label': 'Membrane' if predicted_class == 1 else 'Soluble',
        'confidence': probabilities[predicted_class].item(),
        'probabilities': {
            'Soluble': probabilities[0].item(),
            'Membrane': probabilities[1].item()
        }
    }

# Test on new sequences
new_sequences = [
    generate_protein_sequence(80),   # Short sequence
    generate_protein_sequence(250),  # Long sequence
]

print("Predictions for new sequences:\n")
for i, seq in enumerate(new_sequences, 1):
    result = predict_protein(seq, model, batch_converter, device)
    print(f"Sequence {i} (length {len(seq)}):")
    print(f"  Prediction: {result['predicted_label']}")
    print(f"  Confidence: {result['confidence']:.4f}")
    print(f"  Probabilities: Soluble={result['probabilities']['Soluble']:.4f}, "
          f"Membrane={result['probabilities']['Membrane']:.4f}")
    print()

## Summary

In this notebook, we:

1. ✅ Loaded a pre-trained ESM-2 protein language model
2. ✅ Created a classification head for fine-tuning
3. ✅ Trained the model on protein function prediction
4. ✅ Evaluated performance on a test set
5. ✅ Visualized protein embeddings using t-SNE
6. ✅ Demonstrated prediction on new sequences

**Key Takeaways:**
- Transfer learning from ESM-2 enables effective protein function prediction with limited data
- Feature extraction (frozen ESM-2) works well for many tasks and is computationally efficient
- Fine-tuning (updating ESM-2 weights) can improve performance but requires more data and compute
- ESM-2 embeddings capture meaningful protein properties (structure, function)

**Next Steps:**
- Try fine-tuning with `freeze_esm=False` and compare performance
- Experiment with different ESM-2 model sizes (8M to 15B parameters)
- Apply to real protein function prediction tasks
- Explore attention weights for interpretability